In [ ]:
!pip install -q datasets transformers
!pip install -q sentence-transformers

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving train.csv to train (3).csv


In [ ]:
from datasets import load_dataset
dataset = load_dataset("csv", data_files="train.csv")
train_ds = dataset["train"]
print(train_ds)

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})


Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [ ]:
def combine_columns(example):
    example["combined_text"] = example["prompt"] + " " + example["A"]
    return example

train_ds = train_ds.map(combine_columns)

In [ ]:
row = train_ds[51]

print("Combined Text:")
print(row["combined_text"])

print("\nCharacter Length:")
print(len(row["combined_text"]))

Combined Text:
Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.

Character Length:
614


Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
print(tokenizer.vocab_size)
# or
print(len(tokenizer))

30522
30522


Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.  

In [ ]:
print("SEP Token:", tokenizer.sep_token)
print("SEP Token ID:", tokenizer.sep_token_id)

SEP Token: [SEP]
SEP Token ID: 102


Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors).

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [ ]:
prompts = list(train_ds["prompt"])

encodings = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

print(encodings["input_ids"].shape)

torch.Size([2000, 128])


Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object.

What is the exact shape of the last_hidden_state tensor returned?

Note: We follow zero-indexing here.

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import torch

# Row with zero-based index 0
prompt = train_ds[0]["prompt"]

# Default tokenizer settings (no manual padding or truncation)
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

print("Input IDs shape:", inputs["input_ids"].shape)
print("last_hidden_state shape:", outputs.last_hidden_state.shape)

Input IDs shape: torch.Size([1, 31])
last_hidden_state shape: torch.Size([1, 31, 768])


Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [ ]:
print(round(outputs.last_hidden_state[0, 0, :5].sum().item(), 4))

-1.2001


Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0).

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

*


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [ ]:
text = "Light-ion fusion is a technique."

inputs = tokenizer(text, return_tensors="pt")

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Tokens:")
for i, token in enumerate(tokens):
    print(i, token)

Tokens:
0 [CLS]
1 light
2 -
3 ion
4 fusion
5 is
6 a
7 technique
8 .
9 [SEP]


In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

# Last layer attention
last_layer_attention = outputs.attentions[-1]

# First attention head
head0 = last_layer_attention[0, 0]

# Find the index of "fusion"
fusion_index = tokens.index("fusion")

# Attention from [CLS] (index 0) to "fusion"
attention_weight = head0[0, fusion_index].item()

print("Fusion token index:", fusion_index)
print("Attention weight:", round(attention_weight, 4))

Fusion token index: 4
Attention weight: 0.1025


Context-Aware Embeddings
Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [ ]:
from sentence_transformers import SentenceTransformer, util

model_st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Row with zero-based index 0
prompt = train_ds[0]["prompt"]
option_b = train_ds[0]["B"]

# Generate embeddings
embeddings = model_st.encode(
    [prompt, option_b],
    convert_to_tensor=True
)

# Cosine similarity
similarity = util.cos_sim(embeddings[0], embeddings[1])

print("Cosine Similarity:", round(similarity.item(), 4))

Cosine Similarity: 0.7658


Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set?

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
import numpy as np
from tqdm.auto import tqdm

In [ ]:
def map3_score(actual, predicted):
    score = 0.0

    for a, preds in zip(actual, predicted):
        if a in preds:
            score += 1.0 / (preds.index(a) + 1)

    return score / len(actual)

In [ ]:
actual_answers = []
tfidf_predictions = []
minilm_predictions = []

options = ["A", "B", "C", "D", "E"]

for row in tqdm(train_ds):

    prompt = row["prompt"]
    option_texts = [row[o] for o in options]

    actual_answers.append(row["answer"])

    # -------------------------
    # Pipeline 1 : TF-IDF
    # -------------------------

    documents = [prompt] + option_texts

    vectorizer = TfidfVectorizer()

    tfidf = vectorizer.fit_transform(documents)

    similarities = cosine_similarity(
        tfidf[0:1],
        tfidf[1:]
    )[0]

    ranked = np.argsort(similarities)[::-1]

    tfidf_top3 = [options[i] for i in ranked[:3]]

    tfidf_predictions.append(tfidf_top3)

    # -------------------------
    # Pipeline 2 : MiniLM
    # -------------------------

    embeddings = model_st.encode(
        documents,
        convert_to_tensor=True
    )

    sims = util.cos_sim(
        embeddings[0],
        embeddings[1:]
    )[0].cpu().numpy()

    ranked = np.argsort(sims)[::-1]

    minilm_top3 = [options[i] for i in ranked[:3]]

    minilm_predictions.append(minilm_top3)

  0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
map3 = map3_score(actual_answers, minilm_predictions)

improved = sum(
    (a not in tfidf) and (a in mini)
    for a, tfidf, mini in zip(
        actual_answers,
        tfidf_predictions,
        minilm_predictions
    )
)

print("MiniLM MAP@3:", round(map3, 4))
print("Improved Count:", improved)

MiniLM MAP@3: 0.4231
Improved Count: 564


Zero-shot classification concepts
Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [ ]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification")

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
# Row with zero-based index 1
row = train_ds[1]

sequence = row["prompt"]
candidate_labels = [row["A"], row["B"], row["C"]]

result = classifier(
    sequence,
    candidate_labels=candidate_labels
)

print("Labels:", result["labels"])
print("Scores:", result["scores"])

print("\nTop Probability:", round(result["scores"][0], 4))

Labels: ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to impl

Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True.

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [ ]:
# Row with zero-based index 1
row = train_ds[1]

sequence = row["prompt"]
candidate_labels = [row["A"], row["B"], row["C"]]

result_multi = classifier(
    sequence,
    candidate_labels=candidate_labels,
    multi_label=True
)

print("Labels:", result_multi["labels"])
print("Scores:", result_multi["scores"])

Labels: ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to imple

In [ ]:
# Sum from the previous question (Softmax)
softmax_sum = sum(result["scores"])

# Sum from the current question (Sigmoid)
sigmoid_sum = sum(result_multi["scores"])

difference = abs(softmax_sum - sigmoid_sum)

print("Softmax Sum:", softmax_sum)
print("Sigmoid Sum:", sigmoid_sum)
print("Absolute Difference:", round(difference, 4))

Softmax Sum: 0.9999999403953552
Sigmoid Sum: 0.0005096085387776839
Absolute Difference: 0.9995


In [ ]:
print(result_multi)
print(result_multi["scores"])

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion 

Let's try Generative AI instead of Classification.

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B."
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model?

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer_t5 = AutoTokenizer.from_pretrained("google/flan-t5-small")
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

row = train_ds[0]

text = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} "
    f"or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

inputs = tokenizer_t5(text, return_tensors="pt")

outputs = model_t5.generate(
    **inputs,
    max_new_tokens=5
)

answer = tokenizer_t5.decode(outputs[0], skip_special_tokens=True)

print(repr(answer))

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

'B'


In [ ]:
row = train_ds[0]

text = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} "
    f"or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

result = generator(
    text,
    max_new_tokens=5
)

print(result)
print("\nGenerated text:", repr(result[0]["generated_text"]))